
## 3. Near-Duplicate Detection and Data Leakage Assessment

### 3.1 Objective

This experiment aims to identify potential near-duplicate images across the training, development, and test splits of the CrisisMMD five-class humanitarian classification dataset.

Near-duplicate images across different splits may introduce data leakage, potentially leading to overly optimistic model evaluation results.

The purpose of this experiment is to detect and document potential overlaps before implementing appropriate preprocessing and re-evaluating the models.

### 3.2 Dataset

We use the CrisisMMD v2.0 humanitarian agreed-label subset, following Ofli et al. (2020).

The dataset contains 8,079 image-text samples divided into:

- Training: 6,126 samples
- Development: 998 samples
- Test: 955 samples

The existing splits are retained during this initial assessment.

### 3.3 Methodology

**Perceptual Hashing (pHash)** is used to detect visually similar images.

Unlike conventional file hashing, which identifies exact file matches, pHash generates similar hash values for images with similar visual content, including some resized or compressed versions.

Each image is represented by a 64-bit perceptual hash. Image similarity is measured using the Hamming distance between two hashes.

A smaller Hamming distance indicates greater similarity.

**Detection procedure:**

1. **Load dataset:** Read the humanitarian agreed-label train, development, and test splits.

2. **Generate perceptual hashes:** Calculate a 64-bit pHash for each available image using the `ImageHash` library.

3. **Identify candidate pairs:** Compare image hashes across different splits using a segmented hash index to reduce unnecessary comparisons.

4. **Apply similarity threshold:** Flag image pairs with a Hamming distance of 6 or less as potential near-duplicates.

5. **Check tweet overlap:** Identify tweet IDs that appear in multiple dataset splits.

6. **Export findings:** Save candidate image pairs, their split information, and Hamming distances to a local CSV file for further examination.

The threshold of 6 is used as an initial screening criterion and does not guarantee that every flagged pair is a true duplicate.

### 3.4 Evaluation and Interpretation

After running the code, the following outputs will be examined:

- Number of images successfully processed
- Missing or unreadable image files
- Number of cross-split candidate near-duplicate pairs
- Number of unique test images with potential cross-split matches
- Distribution of perceptual hash distances
- Number of tweet IDs appearing across multiple splits

These outputs will help assess possible overlap between training and evaluation data.

Detected pairs must be visually verified before being classified as confirmed near-duplicates.

Importantly, the number of candidate pairs may differ from the number of unique affected images, since one image can match multiple images.

### 3.5 Limitations

Perceptual hashing is a similarity-screening method rather than a definitive duplicate detection method.

It may produce false positives for visually similar but distinct images, and false negatives when images have undergone substantial cropping, rotation, or other modifications.

The current implementation detects potential cross-split overlaps but does not automatically remove images, modify dataset splits, or retrain models.

### 3.6 Next Steps

Based on the detection results, the following actions may be considered:

1. Manually inspect flagged image pairs.
2. Identify confirmed duplicate groups.
3. Develop a leakage-prevention strategy that preserves an independent evaluation set.
4. Retrain and evaluate affected models after appropriate preprocessing.
5. Compare the original and cleaned-data results using macro-F1 and per-class F1.

No changes to the dataset will be made during this initial detection experiment.

### Reference

Ofli, F., Alam, F., & Imran, M. (2020). Analysis of social media data using multimodal deep learning for disaster response.


In [1]:

# ============================================================
# CrisisMMD: Near-Duplicate Image Detection
# Project 2.1 - Dataset Quality / Leakage Assessment
#
# Task: 5-class humanitarian classification
# Method: Perceptual Hashing (pHash)
#
# This code DOES NOT modify or delete any dataset files.
# ============================================================

%pip -q install ImageHash

import pandas as pd
import imagehash
from PIL import Image
from pathlib import Path
from collections import defaultdict

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

DATA_DIR = Path("../data/CrisisMMD_v2.0")

SPLITS_DIR = (
    Path("../data/splits")
    / "crisismmd_datasplit_agreed_label"
)

OUTPUT_DIR = Path("../cache")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "near_duplicates_humanitarian.csv"

# pHash Hamming distance threshold
# 0 = identical perceptual hash
# <= 6 = candidate near-duplicate
THRESHOLD = 6


# ------------------------------------------------------------
# 2. Load humanitarian agreed-label splits
# ------------------------------------------------------------

frames = []

for split in ["train", "dev", "test"]:

    filename = (
        f"task_humanitarian_text_img_"
        f"agreed_lab_{split}.tsv"
    )

    df = pd.read_csv(
        SPLITS_DIR / filename,
        sep="\t",
        dtype=str,
        quoting=3
    )

    df["split"] = split
    frames.append(df)

data = pd.concat(frames, ignore_index=True)

print("=== Dataset Information ===")
print(data["split"].value_counts())
print("Total samples:", len(data))


# ------------------------------------------------------------
# 3. Calculate perceptual hashes
# ------------------------------------------------------------

print("\nCalculating perceptual hashes...")

hash_cache = {}
records = []
missing = []
corrupt = []

for row in data.itertuples(index=False):

    image_path = DATA_DIR / row.image

    if not image_path.exists():
        missing.append(str(image_path))
        continue

    try:
        if image_path not in hash_cache:

            with Image.open(image_path) as img:
                phash = imagehash.phash(
                    img.convert("RGB"),
                    hash_size=8
                )

            hash_cache[image_path] = int(str(phash), 16)

        records.append({
            "image_id": row.image_id,
            "tweet_id": row.tweet_id,
            "split": row.split,
            "path": str(image_path),
            "hash": hash_cache[image_path]
        })

    except Exception:
        corrupt.append(str(image_path))

print("Images successfully processed:", len(records))
print("Unique files hashed:", len(hash_cache))
print("Missing image files:", len(missing))
print("Unreadable image files:", len(corrupt))


# ------------------------------------------------------------
# 4. Detect cross-split near-duplicates
#
# Divide 64-bit hashes into 7 sections.
# Any pair with Hamming distance <= 6 must share
# at least one identical section.
#
# This avoids comparing every possible image pair.
# ------------------------------------------------------------

print("\nChecking near-duplicates across splits...")

segments = []
offset = 0

for width in [10, 9, 9, 9, 9, 9, 9]:
    segments.append((offset, width))
    offset += width

buckets = defaultdict(list)
matches = []

for i, record in enumerate(records):

    h = record["hash"]
    candidate_indices = set()

    for section, (shift, width) in enumerate(segments):

        part = (h >> shift) & ((1 << width) - 1)
        key = (section, part)

        candidate_indices.update(buckets[key])

    for j in candidate_indices:

        previous = records[j]

        # Focus on potential leakage BETWEEN splits
        if previous["split"] == record["split"]:
            continue

        distance = (h ^ previous["hash"]).bit_count()

        if distance <= THRESHOLD:

            matches.append({
                "split_A": previous["split"],
                "split_B": record["split"],
                "image_id_A": previous["image_id"],
                "image_id_B": record["image_id"],
                "tweet_id_A": previous["tweet_id"],
                "tweet_id_B": record["tweet_id"],
                "path_A": previous["path"],
                "path_B": record["path"],
                "phash_distance": distance
            })

    # Add current hash to the search index
    for section, (shift, width) in enumerate(segments):

        part = (h >> shift) & ((1 << width) - 1)
        buckets[(section, part)].append(i)


# ------------------------------------------------------------
# 5. Summarise results
# ------------------------------------------------------------

results = pd.DataFrame(
    matches,
    columns=[
        "split_A", "split_B",
        "image_id_A", "image_id_B",
        "tweet_id_A", "tweet_id_B",
        "path_A", "path_B",
        "phash_distance"
    ]
)

print("\n=== Near-Duplicate Detection Results ===")

print("Threshold:", THRESHOLD)
print("Cross-split near-duplicate pairs:", len(results))

if not results.empty:

    print("\nPairs by split combination:")

    print(
        results.groupby(
            ["split_A", "split_B"]
        ).size().to_string()
    )

    test_ids = set(
        results.loc[
            results["split_A"] == "test",
            "image_id_A"
        ]
    )

    test_ids.update(
        results.loc[
            results["split_B"] == "test",
            "image_id_B"
        ]
    )

    print(
        "\nUnique test images with cross-split matches:",
        len(test_ids)
    )

    print("\nSample detected distances:")
    print(
        results["phash_distance"]
        .value_counts()
        .sort_index()
        .to_string()
    )

else:
    print("No cross-split matches at this threshold.")


# ------------------------------------------------------------
# 6. Check tweet-ID overlap across splits
# ------------------------------------------------------------

tweet_overlap = (
    data.groupby("tweet_id")["split"]
    .nunique()
)

overlapping_tweets = int(
    (tweet_overlap > 1).sum()
)

print(
    "\nTweet IDs occurring in multiple splits:",
    overlapping_tweets
)


# ------------------------------------------------------------
# 7. Save results locally
# ------------------------------------------------------------

results.to_csv(OUTPUT_FILE, index=False)

print("\n=== Completed ===")
print("CSV saved to:", OUTPUT_FILE)
print("Original dataset was NOT modified.")
print("Manually verify flagged image pairs.")
print("Keep the CSV local; do not upload raw IDs or paths.")



[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


=== Dataset Information ===
split
train    6126
dev       998
test      955
Name: count, dtype: int64
Total samples: 8079

Calculating perceptual hashes...


Images successfully processed: 8079
Unique files hashed: 8079
Missing image files: 0
Unreadable image files: 0

Checking near-duplicates across splits...

=== Near-Duplicate Detection Results ===
Threshold: 6
Cross-split near-duplicate pairs: 287

Pairs by split combination:
split_A  split_B
dev      test        26
train    dev        153
         test       108

Unique test images with cross-split matches: 52

Sample detected distances:
phash_distance
0    116
2     40
4     36
6     95

Tweet IDs occurring in multiple splits: 0

=== Completed ===
CSV saved to: ../cache/near_duplicates_humanitarian.csv
Original dataset was NOT modified.
Manually verify flagged image pairs.
Keep the CSV local; do not upload raw IDs or paths.
